In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("naiyakhalid/flood-prediction-dataset")

print("Path to dataset files:", path)

100%|██████████| 28.6M/28.6M [00:00<00:00, 137MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/naiyakhalid/flood-prediction-dataset/versions/2


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("path").getOrCreate()
data=spark.read.csv(path,header=True,inferSchema=True)
data.show()

+---+----------------+------------------+---------------+-------------+------------+-------------+-----------+---------+---------------------+-------------+-------------------------------+---------------+--------------------+----------+----------+---------------------------+---------------+-----------+------------------+----------------+----------------+
| id|MonsoonIntensity|TopographyDrainage|RiverManagement|Deforestation|Urbanization|ClimateChange|DamsQuality|Siltation|AgriculturalPractices|Encroachments|IneffectiveDisasterPreparedness|DrainageSystems|CoastalVulnerability|Landslides|Watersheds|DeterioratingInfrastructure|PopulationScore|WetlandLoss|InadequatePlanning|PoliticalFactors|FloodProbability|
+---+----------------+------------------+---------------+-------------+------------+-------------+-----------+---------+---------------------+-------------+-------------------------------+---------------+--------------------+----------+----------+---------------------------+-----------

In [7]:
data.printSchema()

root
 |-- id: string (nullable = true)
 |-- MonsoonIntensity: string (nullable = true)
 |-- TopographyDrainage: string (nullable = true)
 |-- RiverManagement: string (nullable = true)
 |-- Deforestation: string (nullable = true)
 |-- Urbanization: string (nullable = true)
 |-- ClimateChange: string (nullable = true)
 |-- DamsQuality: string (nullable = true)
 |-- Siltation: string (nullable = true)
 |-- AgriculturalPractices: string (nullable = true)
 |-- Encroachments: string (nullable = true)
 |-- IneffectiveDisasterPreparedness: string (nullable = true)
 |-- DrainageSystems: string (nullable = true)
 |-- CoastalVulnerability: string (nullable = true)
 |-- Landslides: string (nullable = true)
 |-- Watersheds: string (nullable = true)
 |-- DeterioratingInfrastructure: string (nullable = true)
 |-- PopulationScore: string (nullable = true)
 |-- WetlandLoss: string (nullable = true)
 |-- InadequatePlanning: string (nullable = true)
 |-- PoliticalFactors: string (nullable = true)
 |-- Fl

In [9]:
input=["MonsoonIntensity", "TopographyDrainage", "RiverManagement", "Deforestation", "Urbanization", "ClimateChange", "DamsQuality", "Siltation", "AgriculturalPractices", "Encroachments", "IneffectiveDisasterPreparedness", "DrainageSystems", "CoastalVulnerability", "Landslides", "Watersheds", "DeterioratingInfrastructure", "PopulationScore", "WetlandLoss", "InadequatePlanning", "PoliticalFactors"]
output=["FloodProbability"]

In [5]:
from pyspark.ml.feature import StringIndexer,OneHotEncoder,VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline

In [12]:
stages = []
encoded_features = []


data_cleaned = data.na.drop(subset=[output[0]])

for col_name in input:

    indexer = StringIndexer(inputCol=col_name, outputCol=col_name + "_indexed", handleInvalid="keep")
    stages.append(indexer)


    encoder = OneHotEncoder(inputCol=col_name + "_indexed", outputCol=col_name + "_vec", handleInvalid="keep")
    stages.append(encoder)
    encoded_features.append(col_name + "_vec")


assembler = VectorAssembler(inputCols=encoded_features, outputCol="features")
stages.append(assembler)


lr = LinearRegression(featuresCol="features", labelCol=output[0])
stages.append(lr)


pipeline = Pipeline(stages=stages)

# Fit the model using the cleaned data
model = pipeline.fit(data_cleaned)

In [13]:
# Make predictions on the cleaned data
predictions = model.transform(data_cleaned)

# Display some of the predictions
predictions.select(output[0], "prediction").show()

+----------------+-------------------+
|FloodProbability|         prediction|
+----------------+-------------------+
|           0.445|0.47775322844546564|
|            0.45| 0.4762529482281097|
|            0.53| 0.5057930773391232|
|           0.535| 0.5336387315350318|
|           0.415| 0.3529662057360471|
|            0.44|0.44988892976060174|
|            0.46|0.45441578189791093|
|           0.595| 0.5459403982078525|
|           0.505| 0.4676891304563406|
|           0.455|0.46619660478089503|
|           0.515|0.47813440777386074|
|            0.48|0.47107298978879447|
|            0.47|0.47638292858974957|
|            0.51|0.46611035580204974|
|           0.485| 0.4658517605829556|
|            0.43|0.45491855804630066|
|           0.525|  0.529018861938365|
|           0.515|0.46472531447644216|
|            0.56| 0.5513608470044903|
|           0.555| 0.5457655619332835|
+----------------+-------------------+
only showing top 20 rows


### Model Evaluation

Let's evaluate the performance of our Linear Regression model using metrics like Root Mean Squared Error (RMSE) and R-squared. These metrics help us understand how well our model's predictions fit the actual values.

In [18]:
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize a RegressionEvaluator
evaluator_rmse = RegressionEvaluator(labelCol=output[0], predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol=output[0], predictionCol="prediction", metricName="r2")

# Calculate RMSE
rmse = evaluator_rmse.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on test data = {rmse}")

# Calculate R-squared
r2 = evaluator_r2.evaluate(predictions)
print(f"R-squared on test data = {r2}")

Root Mean Squared Error (RMSE) on test data = 0.020042408231950613
R-squared on test data = 0.8457179062678639
